# Figure: correlation-function estimator validation

Loads `results/corrfunc/validation.json` (`bench/corrfunc/validate_vs_baseline.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

_here = Path.cwd()
_candidates = [
    _here / "results" / "corrfunc",
    _here.parents[1] / "results" / "corrfunc",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "validation.json").read_text())
meta = data["metadata"]
records = data["records"]
thetas = sorted({r["theta"] for r in records})
sizes = sorted({r["n"] for r in records})
print(f"device={meta['device_kind']} jax={meta['jax_version']} commit={meta.get('git_commit')}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
for i, n in enumerate(sizes):
    xs, ys = [], []
    for th in thetas:
        rec = next((r for r in records if r["n"] == n and r["theta"] == th), None)
        if rec is None:
            continue
        xs.append(th)
        ys.append(max(rec["max_per_bin_rel_error"], 1e-16))
    ax.plot(xs, ys, "o-", color=PALETTE[i % len(PALETTE)], label=f"N={n}")
ax.set_yscale("log")
ax.set_xlabel(r"opening angle $\theta$")
ax.set_ylabel("max per-bin rel. error vs. brute force")
ax.set_title("Tree estimator accuracy vs. exact soft counts")
ax.legend()
out = FIGDIR / "fig_corrfunc_validation.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
